# Geospatial Visualization

Welcome to the Advanced tier of visualization! Standard charts are great for abstract numbers, but when your data contains locations (cities, zip codes, or exact latitude/longitude coordinates), forcing that data into a bar chart is a missed opportunity. 

Geospatial visualization unlocks the "Where" in your data. In this lesson, we are going to leave `matplotlib` behind and use **Plotly**, an interactive graphing library that allows us to build zoomable, clickable maps right inside our notebook.

If you are analyzing a supply chain, real estate portfolios, or delivery routes, location is often the single most predictive feature you have. 

There are three primary ways we visualize data on a map:
1. **Point Maps**: Plotting exact coordinates (Lat/Lon) to show specific locations.
2. **Density Maps (Heatmaps)**: Blurring individual points together to show the concentration or "hotspots" of an activity.
3. **Choropleth Maps**: Coloring entire geographic borders (like states or countries) based on an aggregated metric.

Let's set up a Python sandbox. *(Note: You will need to install Plotly by running `pip install plotly` in your terminal if you haven't already).*

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

# 1. Create a simulated dataset for Retail Store Locations (Point/Density Data)
store_data = {
    'Store_ID': [1, 2, 3, 4, 5, 6, 7],
    'City': ['New York', 'Boston', 'Philadelphia', 'Los Angeles', 'San Diego', 'San Francisco', 'Chicago'],
    'Lat': [40.7128, 42.3601, 39.9526, 34.0522, 32.7157, 37.7749, 41.8781],
    'Lon': [-74.0060, -71.0589, -75.1652, -118.2437, -117.1611, -122.4194, -87.6298],
    'Annual_Revenue': [5.2, 3.1, 2.8, 6.5, 3.0, 4.8, 4.0], # In Millions
    'Avg_Customer_Age': [32, 28, 35, 29, 31, 34, 38]
}
df_stores = pd.DataFrame(store_data)

# 2. Create a simulated dataset for State-Level Logistics (Choropleth Data)
# We will just generate some random package delay times for 5 US states
state_data = {
    'State_Code': ['NY', 'CA', 'IL', 'TX', 'FL'],
    'Avg_Delay_Hours': [4.5, 12.0, 2.1, 8.4, 5.0]
}
df_states = pd.DataFrame(state_data)

print("✅ Geospatial Datasets loaded!")
display(df_stores.head(3))

✅ Geospatial Datasets loaded!


,Store_ID,City,Lat,Lon,Annual_Revenue,Avg_Customer_Age
0,1,New York,40.7128,-74.0060,5.2,32
1,2,Boston,42.3601,-71.0589,3.1,28
2,3,Philadelphia,39.9526,-75.1652,2.8,35


# 1. Point Maps (Scatter Mapbox)
The most fundamental map is the Point Map. You use this when you have exact Latitude and Longitude coordinates and you want to plot specific entities (like stores, delivery trucks, or individual houses).

To make the map informative, we map our data to visual pre-attentive attributes:
* **Size**: Let's make the size of the bubble represent `Annual_Revenue`.
* **Color**: Let's make the color of the bubble represent `Avg_Customer_Age`.

In [2]:
# Create an interactive Point Map
fig_points = px.scatter_mapbox(
    df_stores, 
    lat="Lat", 
    lon="Lon", 
    hover_name="City",       # Text that appears when you hover your mouse
    size="Annual_Revenue",   # Bigger bubbles = More Revenue
    color="Avg_Customer_Age",# Color gradient based on age
    color_continuous_scale=px.colors.sequential.Plasma,
    size_max=25,             # Cap the maximum bubble size
    zoom=3,                  # Default zoom level (1=World, 20=Street)
    mapbox_style="carto-positron", # Use a clean, light base map
    title="Retail Store Locations (Size = Revenue, Color = Avg Age)"
)

# Render the interactive map
fig_points.show()

/tmp/ipykernel_10940/1705501141.py:2: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_points = px.scatter_mapbox(


*(Insight: Because this is Plotly, you can actually click and drag to pan around, and scroll to zoom in. You can instantly see the cluster on the East Coast and notice how the Los Angeles bubble is massively larger than the others, indicating it is our flagship store.)*

# 2. Density Heatmaps (Geospatial Hotspots)
If you have 10,000 delivery drop-offs in a city, a Point Map becomes a giant, unreadable blob of overlapping dots. 

To solve this, we use a **Density Map**. Instead of showing individual points, it groups them together and creates a "hotspot" gradient. This is perfect for identifying where to build your next warehouse based on customer demand.

In [3]:
# Create an interactive Density Map
fig_density = px.density_mapbox(
    df_stores, 
    lat="Lat", 
    lon="Lon", 
    z="Annual_Revenue",      # The 'weight' of the heatmap
    radius=40,               # How far the heat spreads from the point
    zoom=3, 
    mapbox_style="stamen-terrain", # Use a topographical base map
    title="Revenue Density Hotspots"
)

fig_density.show()

/tmp/ipykernel_10940/4110937223.py:2: DeprecationWarning:

*density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



# 3. Choropleth Maps (Area Aggregations)
What if you don't have exact Lat/Lon coordinates? What if you only have a column that says "Texas" or "CA"? 

You use a **Choropleth Map**. This map takes predefined geographic boundaries (like states, countries, or zip codes) and colors the entire shape based on a metric. 

*Note: For complex, custom shapes (like specific voting districts), you would need to provide a geometric blueprint file called a `GeoJSON`. Fortunately, Plotly has USA States and World Countries built-in!*

In [4]:
# Create an interactive Choropleth Map
fig_choro = px.choropleth(
    df_states,
    locations="State_Code",     # The column with our state abbreviations
    locationmode="USA-states",  # Tell Plotly to use its built-in USA state borders
    color="Avg_Delay_Hours",    # Color the state based on this metric
    scope="usa",                # Restrict the map view to just the USA
    color_continuous_scale="Reds", # Use a sequential Red palette (since delays are bad!)
    title="Average Package Delay by State (Hours)"
)

fig_choro.show()

*(Insight: This is the ultimate executive summary chart. At a single glance, the leadership team can see a massive, dark red block over California, instantly identifying a severe logistical bottleneck on the West Coast that needs attention.)*

## Real-World Use Case or Analogy:
Think of Geospatial Visualization in the context of a **National Logistics and Freight Company**:

* **The Problem (Standard Charts)**: The supply chain manager looks at a bar chart showing "Package Delays by Region." It shows that the "Midwest" has a 20% delay rate. It's helpful, but "Midwest" is a massive area. The chart doesn't tell them exactly *where* to send help.
* **Point Maps (The Fleet View)**: The manager switches to a map with exact Lat/Lon plots of every broken-down delivery truck. They immediately see 15 dots clustered along a specific 10-mile stretch of Interstate 80 in Wyoming. 
* **Density Map (The Weather Overlay)**: By switching to a density heatmap and overlaying weather data, the manager realizes those 15 dots perfectly align with a massive blizzard hotspot. The trucks aren't broken; they are snowed in. 
* **Choropleth (The Executive Briefing)**: The manager needs to ask the CEO for budget to buy snow-tires. They don't show the CEO 15 dots on a highway. They show a Choropleth map of the US, with Wyoming colored dark red for "Lost Revenue." The visual impact secures the budget instantly. 

---
